# 03 · Middleware 洋葱模型

目标：用标准库 handler wrapper 观察请求进入与响应退出顺序。对应文档：`03_architecture_patterns/3.2-middleware-and-pipeline.md`。

## 从 Koa/Express 迁移

Go middleware 通常是 `func(http.Handler) http.Handler`。包装顺序决定进入顺序，`defer` 决定反向退出，形状接近 Koa 洋葱模型。

In [ ]:
import (
    "fmt"
    "net/http"
    "net/http/httptest"
)

In [ ]:
func Trace(name string, next http.Handler) http.Handler {
    return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
        fmt.Println("enter", name)
        defer fmt.Println("exit ", name)
        next.ServeHTTP(w, r)
    })
}

## 先预测调用顺序

对 `Trace(outer, Trace(inner, endpoint))`，先写下五行日志的顺序，再运行实验。

In [ ]:
%%
endpoint := http.HandlerFunc(func(w http.ResponseWriter, _ *http.Request) {
    fmt.Println("handler")
    w.WriteHeader(http.StatusNoContent)
})
handler := Trace("outer", Trace("inner", endpoint))

request := httptest.NewRequest(http.MethodGet, "/", nil)
response := httptest.NewRecorder()
handler.ServeHTTP(response, request)
fmt.Println("status", response.Code)

进入顺序是 outer → inner → handler；退出顺序是 inner → outer。认证、限流、recover、日志和 timeout 的相对位置会改变系统语义。

## 直接调用本地 module

把 Labs module 映射进 GoNB 的临时 module 后，可以直接 import 非 `internal` 包。Go 的 `internal` 规则仍然生效。

In [ ]:
!*cd "$GONB_TMP_DIR" && go mod edit -require=example.com/architecture-labs@v0.0.0 -replace=example.com/architecture-labs="$GO_LEARNING_LABS"

In [ ]:
import gatewaylab "example.com/architecture-labs/gateway"

In [ ]:
%%
labRouter := gatewaylab.NewRouter([]gatewaylab.Provider{
    {Name: "primary", Healthy: false},
    {Name: "backup", Healthy: true},
})
selected, err := labRouter.Select("primary")
fmt.Printf("selected=%s err=%v\n", selected.Name, err)

In [ ]:
!*cd "$GO_LEARNING_CAPSTONE" && go test ./internal/gateway/core -run TestEndpointMiddleware -v

## 动手挑战

1. 增加一个写入 `X-Request-ID` 的 middleware。
2. 增加 recover middleware，并比较放在 Trace 内外时的日志差异。
3. 调换 outer/inner，解释为什么代码阅读时要从组装点确认顺序。

## 复盘

- middleware 为什么是一种装饰器？
- recover 应覆盖哪些组件？
- HTTP 边界策略与领域业务规则应如何分工？